# Domain Decider → Research Module

Open this project in its VS Code Dev Container and select **SedAI Docker — Python 3.12** (`/usr/local/bin/python`).

One cell runs both phases. Blank resume creates `runs/<factsheet>/run_001/`; set an explicit full-run root to resume with frozen inputs/settings. Do not run the same job in two environments. Review public-input consent before execution. Advanced linked/upload-only actions remain in the module APIs and CLI.


In [1]:
from pathlib import Path

from ML.deep_research.workflow import create_run, run_all
from ML.deep_research.domain_decider.backend.fs import load_json
from ML.deep_research.domain_decider.backend.settings import DOMAIN_PLUGIN_PATH, RUNS_DIR

FACT_SHEETS_DIR = Path("inputs/facts")
DOMAIN_PLUGIN = DOMAIN_PLUGIN_PATH
REQUIREMENTS_PATH = Path("inputs/requirement.md")
SOURCE_SUGGESTION_PATH = Path("inputs/source_suggestion.md")
RESEARCH_INSTRUCTION_PATH = Path("inputs/user_research_instruction.md")
RESEARCH_CONFIG_PATH = Path("inputs/research_config.json")

RESUME_RUN_PATH = ""  # Explicit full-run root; blank starts fresh runs for every factsheet
PUBLIC_INPUT_CONFIRMED = True
REASONING_SUMMARIES = True
RESEARCH_FACTSHEET_ACCESS = False  # False disables original-factsheet access; new runs only
RETRY_FAILED = False

# Reasoning: low | medium | high | max; verbosity/search_context: low | medium | high
STAGE_SETTINGS = {
    "metadata": {"reasoning": "high", "verbosity": "medium"},
    "design": {"reasoning": "high", "search_context": "medium", "verbosity": "medium"},
    "distribution": {"reasoning": "high", "verbosity": "medium"},
    "source_discovery": {"reasoning": "high", "search_context": "medium", "verbosity": "low"},
    "research": {"reasoning": "high", "verbosity": "medium"},
    "research_search": {"reasoning": "high", "search_context": "medium", "verbosity": "low"},
    "document": {"reasoning": "high", "verbosity": "medium"},
    "summary": {"reasoning": "medium", "verbosity": "medium"},
}

if RETRY_FAILED and not RESUME_RUN_PATH.strip():
    raise ValueError("RETRY_FAILED requires an explicit resume path.")

if RESUME_RUN_PATH.strip():
    queue = [Path(RESUME_RUN_PATH)]
    print("Resume uses saved factsheet access; the notebook toggle applies only to new runs.")
else:
    if not FACT_SHEETS_DIR.is_dir():
        raise ValueError("FACT_SHEETS_DIR must be an existing folder.")
    queue = sorted((p for p in FACT_SHEETS_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".md"),
                   key=lambda p: (p.name.casefold(), p.name))
    if not queue:
        raise ValueError("FACT_SHEETS_DIR contains no Markdown factsheets.")

BATCH_RESULTS = []
try:
    for index, source in enumerate(queue, 1):
        print(f"Factsheet {index}/{len(queue)}: {source.name}")
        FULL_RUN = None
        outcome = {"file": source.name, "status": "interrupted", "run": None, "reports": None}
        BATCH_RESULTS.append(outcome)
        try:
            FULL_RUN = source if RESUME_RUN_PATH.strip() else create_run(
                source, domain_plugin=DOMAIN_PLUGIN, requirements=REQUIREMENTS_PATH,
                source_suggestion=SOURCE_SUGGESTION_PATH, research_instruction=RESEARCH_INSTRUCTION_PATH,
                research_config=RESEARCH_CONFIG_PATH, runs_dir=RUNS_DIR,
                research_factsheet_access=RESEARCH_FACTSHEET_ACCESS,
                stage_settings={**STAGE_SETTINGS, "reasoning_summaries": REASONING_SUMMARIES},
                public_input_confirmed=PUBLIC_INPUT_CONFIRMED,
            )
            outcome["run"] = FULL_RUN
            print(f"Run: {FULL_RUN}\nLog: {FULL_RUN / 'run.log'}")
            await run_all(FULL_RUN, retry_failed=RETRY_FAILED)
            outcome["status"] = "finished"
        except Exception as error:
            outcome["status"] = "failed"
            print(f"Workflow failed ({type(error).__name__}); see its log if a run was created.")
        # Cancellation is not caught: stop the queue and keep completed outcomes.
        if FULL_RUN is None:
            continue
        try:
            saved = load_json(FULL_RUN / "run.json")
            print(f"Workflow: {saved['status']}")
            for phase, relative in saved.get("phases", {}).items():
                phase_path = FULL_RUN / relative
                if not (phase_path / "run.json").exists():
                    print(f"{phase}: not started")
                    continue
                state = load_json(phase_path / "run.json")
                print(f"{phase}: {state['status']} — {phase_path}")
                if phase == "research_module":
                    print(f"Original factsheet access: {state.get('research', {}).get('factsheet', {}).get('status', 'not enabled by historical policy')} (saved)")
                    print(f"Discovery: {state.get('discovery_status', 'pending')}")
                    uploads = state.get("document_uploads", {})
                    print(f"Uploads: {uploads.get('status', 'pending')}; {uploads.get('counts', {})}")
                    jobs = state.get("research", {}).get("jobs", {})
                    print(f"Research: {sum(j.get('status') == 'complete' for j in jobs.values())}/{len(state['domains'])}")
                    for job, entry in jobs.items():
                        budget = entry.get("budget", {})
                        left = budget.get("remaining")
                        balance = 'not recorded' if 'remaining' not in budget else ('unlimited' if left is None else left)
                        print(f"{job}: {entry['status']}; {budget.get('used', 'not recorded')} used; {balance} remaining")
                    print(f"Sources: {phase_path / 'sources'}\nReports: {phase_path / 'research'}")
            if outcome["status"] != "failed":
                outcome["status"] = saved["status"]
            outcome["reports"] = FULL_RUN / saved["phases"]["research_module"] / "research"
        except Exception as error:
            outcome["status"] = "failed"
            print(f"Unable to read saved status ({type(error).__name__}); preserve the run for inspection.")
finally:
    print("\nBatch summary (explicit resume is per run; blank resume creates fresh runs):")
    for outcome in BATCH_RESULTS:
        print(f"{outcome['file']}: {outcome['status']} | Reports: {outcome['reports'] or 'not available'}"
              f" | RESUME_RUN_PATH: {outcome['run'] or 'no run created'}")


Factsheet 1/5: 08e66f65-c3c4-5bbc-a2b3-b4f5246d8a33.md
Run: /app/runs/08e66f65-c3c4-5bbc-a2b3-b4f5246d8a33/run_001
Log: /app/runs/08e66f65-c3c4-5bbc-a2b3-b4f5246d8a33/run_001/run.log
Workflow: complete
domain_decider: complete — /app/runs/08e66f65-c3c4-5bbc-a2b3-b4f5246d8a33/run_001/domain_decider
research_module: complete — /app/runs/08e66f65-c3c4-5bbc-a2b3-b4f5246d8a33/run_001/research_module
Original factsheet access: disabled (saved)
Discovery: complete
Uploads: partial; {'candidate_entries': 54, 'unique_urls': 50, 'uploaded_files': 36, 'failed_urls': 14, 'observations': 0}
Research: 12/12
source_finder/000001: complete; 29 used; 51 remaining
source_finder/000002: complete; 18 used; 62 remaining
source_finder/000003: complete; 37 used; 43 remaining
source_finder/000004: complete; 22 used; 58 remaining
source_finder/000005: complete; 18 used; 62 remaining
source_finder/000006: complete; 23 used; 57 remaining
source_finder/000007: complete; 22 used; 58 remaining
source_finder/000008:

/usr/local/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ResponseOutputMessage` - serialized value may not be as expected [field_name='output', input_value=ResponseFunctionWebSearch... type='web_search_call'), input_type=ResponseFunctionWebSearch])
  PydanticSerializationUnexpectedValue(Expected `ResponseFileSearchToolCall` - serialized value may not be as expected [field_name='output', input_value=ResponseFunctionWebSearch... type='web_search_call'), input_type=ResponseFunctionWebSearch])
  PydanticSerializationUnexpectedValue(Expected `ResponseFunctionToolCall` - serialized value may not be as expected [field_name='output', input_value=ResponseFunctionWebSearch... type='web_search_call'), input_type=ResponseFunctionWebSearch])
  PydanticSerializationUnexpectedValue(Expected `ResponseFunctionToolCallOutputItem` - serialized value may not be as expected [field_name='output', input_value=Res

Workflow: complete
domain_decider: complete — /app/runs/8ac7fdc4-6b3a-52c0-a1af-66aab0b02b3b/run_001/domain_decider
research_module: complete — /app/runs/8ac7fdc4-6b3a-52c0-a1af-66aab0b02b3b/run_001/research_module
Original factsheet access: disabled (saved)
Discovery: complete
Uploads: partial; {'candidate_entries': 49, 'unique_urls': 41, 'uploaded_files': 38, 'failed_urls': 2, 'observations': 0}
Research: 11/11
source_finder/000001: complete; 24 used; 56 remaining
source_finder/000002: complete; 25 used; 55 remaining
source_finder/000003: complete; 18 used; 62 remaining
source_finder/000004: complete; 24 used; 56 remaining
source_finder/000005: complete; 14 used; 66 remaining
source_finder/000006: complete; 15 used; 65 remaining
source_finder/000007: complete; 29 used; 51 remaining
source_finder/000008: complete; 25 used; 55 remaining
source_finder/000009: complete; 23 used; 57 remaining
source_finder/000010: complete; 26 used; 54 remaining
source_finder/000011: complete; 19 used; 61